# Radar GIF Maker

This code will read through a data directory and generate a GIF given all of the radar data in the directory.

In [5]:
import pyart
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import cartopy.feature as cfeature
import xarray as xr
import os
import imageio.v2 as imageio
import pandas as pd
import time
import io
import cmweather
from datetime import datetime

In [3]:
citation_nav = pd.read_csv('flight_data/2011-05-20_CITATION.dat', sep = ',', header = None)
cit_time = citation_nav.iloc[:, 1]
cit_lat = citation_nav.iloc[:, 2]
cit_lon = citation_nav.iloc[:, 3]

In [10]:
with imageio.get_writer('radar_images/May_20_2011_C-SAPR_aircraft.gif', mode='I', loop = 0, duration = 500) as writer:
    with os.scandir('C-SAPR_May_20_2011/') as entries:
        for entry in entries:
            # Data read in 
            filename = entry.name
            data = pyart.io.read('C-SAPR_May_20_2011/' + filename)
                
            # Reflectivity plotting 
            display = pyart.graph.RadarMapDisplay(data)
                        
            projection = ccrs.PlateCarree()
            
            fig = plt.figure(figsize = (11,9))
                        
            # Reflectivity plot
            ax = fig.add_axes([0.06, 0.50, 0.40, 0.40], projection = projection)
            ax.add_feature(cfeature.STATES.with_scale('10m'), linewidth = 1.5, edgecolor = 'black')
                        
            display.plot_ppi_map('corrected_reflectivity_horizontal', 
                                ax = ax,
                                sweep = 1, 
                                vmin = -30, 
                                vmax = 70,
                                cmap = 'ChaseSpectral') #Variable name
                        
            # Velocity plot
            ax1 = fig.add_axes([0.53, 0.50, 0.40, 0.40], projection = projection)
            ax1.add_feature(cfeature.STATES.with_scale('10m'), linewidth = 1.5, edgecolor = 'black')
                        
            display.plot_ppi_map('mean_doppler_velocity', 
                                ax = ax1, 
                                sweep = 1, 
                                vmin = -30,
                                vmax = 30,
                                cmap = 'balance')
                        
            '''
            Reference table for different changable variables 
                        
                                    sweep = 0, #Elevation angle
                                    vmin = -20, #Min value
                                    vmax = 80, #Max value
                                    min_lon = min_lon, #Min longitude of plot
                                    max_lon = max_lon, #Max longitude of plot
                                    min_lat = min_lat, #Min latitude of plot
                                    max_lat = max_lat, #Max latitude of plot
                                    resolution = '10m', #Projection resolution
                                    projection = projection, #Set projection
                                    colorbar_flag = 0, #Turn off colorbar; will add manually
                                    title_flag = 0, #Turn off title; will add manually
                                    ax = ax, # Display axis set
                                    fig = fig, #Where to plot data
                                    embellish = False, #Turn off default map outlines
                                    lat_0 = center_lat, #Center latitude
                                    lon_0 = center_lon, #Center longitude
                                    cmap = 'NWSRef' #Color scheme for colorbar
                                             
            '''
                        
            # Code to add aircraft marker to represent location of aircraft during in-situ measurements 
            #arcft_pos = (-97.5, 36.5)
            #ax.annotate('\u2708', xy = arcft_pos, fontsize = 12, rotation = 90)
            #ax1.annotate('\u2708', xy = arcft_pos, fontsize = 12, rotation = 90)
            
            time_dif = []
            for time in range(len(citation_nav)):
                time_dif.append(abs(datetime.strptime(data.time['units'].split()[2], "%Y-%m-%dT%H:%M:%SZ").timestamp() - datetime.strptime(cit_time[time][0:19], "%Y-%m-%dT%H:%M:%S").timestamp()))
            
            min_time = time_dif.index(min(time_dif))

            arcft_pos = (cit_lon[min_time], cit_lat[min_time])
            ax.annotate('\u2708', xy = arcft_pos, fontsize = 12, ha = 'center', va = 'center')
            ax1.annotate('\u2708', xy = arcft_pos, fontsize = 12, ha = 'center', va = 'center')
            
            
            plt.tight_layout()
            
            # Temporary saving of matplotlib file 
            buf = io.BytesIO()
            fig.savefig(buf, format="png", bbox_inches="tight")
            buf.seek(0)
            writer.append_data(imageio.imread(buf))
            buf.close()
            plt.close(fig)